## 1. 전체 구조

현재 Research Agent는 다음 흐름으로 구성되어 있습니다.

```text
사용자 질문
  -> classify_research_route()
  -> DB / Graph / Web 검색 여부 결정
  -> run_research()
  -> DB RAG 실행
  -> 필요하면 Graph RAG 포함
  -> 필요하면 Web RAG 실행
  -> merge_retrieved_documents()
  -> build_research_context()
  -> AgentState에 retrieved_docs, context 저장
```


## 2. 준비: 프로젝트 루트 찾기

노트북을 어디에서 실행하든 프로젝트 루트를 찾아 `sys.path`에 추가합니다. 그래야 `src.agents.research` 같은 프로젝트 모듈을 import할 수 있습니다.

In [ ]:
from pathlib import Path
import sys


def find_project_root(start: Path) -> Path:
    """README.md와 src 폴더가 있는 위치를 프로젝트 루트로 봅니다."""
    for path in [start, *start.parents]:
        if (path / "README.md").exists() and (path / "src").exists():
            return path
    raise RuntimeError("프로젝트 루트를 찾지 못했습니다.")


project_root = find_project_root(Path.cwd())
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

project_root

## 3. Research Agent import 확인

먼저 새로 만든 Research Agent 모듈을 import합니다.

In [ ]:
from src.agents.research import (
    classify_research_route,
    research_agent,
    run_research,
)
from src.agents.research.agent import (
    build_research_context,
    merge_retrieved_documents,
)

print("Research Agent import OK")

## 4. 라우팅 규칙 검증

`classify_research_route()`는 질문을 보고 어떤 검색기를 사용할지 결정합니다.

- DB RAG: 기본으로 항상 사용
- Graph RAG: 보스, 장비, 세트효과, 요구 스펙처럼 관계형 정보가 필요할 때 사용
- Web RAG: 최신, 공지, 이벤트, 패치처럼 최신성이 중요할 때 사용

아래 셀은 여러 질문을 넣고 라우팅 결과를 표처럼 확인합니다.

In [ ]:
sample_questions = [
    "아델 6차 강화 우선순위 알려줘",
    "노말 스우 요구 스펙 알려줘",
    "이번 하이퍼 버닝 이벤트 보상 알려줘",
    "카루타 세트효과 알려줘",
    "최신 패치에서 보스 보상 바뀐 거 있어?",
]

for question in sample_questions:
    route = classify_research_route(question)
    print("질문:", question)
    print("  use_db:", route["use_db"])
    print("  use_graph:", route["use_graph"])
    print("  use_web:", route["use_web"])
    print("  reason:", route["reason"])
    print()

## 5. 문서 병합 로직 검증

검색 결과는 DB, Graph, Web에서 따로 들어올 수 있습니다. 같은 문서가 여러 번 들어오면 답변 근거가 중복되어 지저분해집니다.

`merge_retrieved_documents()`는 다음 기준으로 문서를 정리합니다.

1. `chunk_id`, `document_id`, `source_url`, `url` 중 하나를 문서 고유값으로 사용합니다.
2. 같은 문서가 중복되면 신뢰도와 점수가 더 높은 문서를 남깁니다.
3. 최종 문서는 신뢰도, 최신성, 검색 점수 순서로 정렬합니다.

In [ ]:
fake_docs = [
    {
        "page_content": "첫 번째 스우 요구 스펙 문서입니다.",
        "metadata": {
            "chunk_id": "boss-swoo-1",
            "title": "스우 요구 스펙",
            "source_url": "graph://boss/swoo",
            "reliability": "graph_seed",
            "retrieval_method": "graph",
        },
        "score": 0.70,
        "source": "graph://boss/swoo",
    },
    {
        "page_content": "더 높은 점수의 스우 요구 스펙 문서입니다.",
        "metadata": {
            "chunk_id": "boss-swoo-1",
            "title": "스우 요구 스펙",
            "source_url": "graph://boss/swoo",
            "reliability": "graph_seed",
            "retrieval_method": "graph",
        },
        "score": 0.95,
        "source": "graph://boss/swoo",
    },
    {
        "page_content": "공식 이벤트 보상 문서입니다.",
        "metadata": {
            "url": "https://maplestory.nexon.com/news/event/1",
            "title": "이벤트 보상",
            "reliability": "HIGH",
            "freshness": "HIGH",
            "retrieval_method": "web",
        },
        "score": 0.50,
        "source": "https://maplestory.nexon.com/news/event/1",
    },
]

merged_docs = merge_retrieved_documents(fake_docs)
print("병합 전 문서 수:", len(fake_docs))
print("병합 후 문서 수:", len(merged_docs))

for doc in merged_docs:
    print(doc["metadata"].get("title"), doc["score"], doc["page_content"])

## 6. context 생성 검증

`context`는 Final Answer Agent가 읽는 근거 묶음입니다. 사람이 보기에도 이해 가능해야 하고, 출처와 신뢰도도 함께 들어가야 합니다.

In [ ]:
context = build_research_context(merged_docs)
print(context)

## 7. Docker / DB 상태 확인

실제 RAG 검색을 하려면 PostgreSQL과 Neo4j에 데이터가 들어 있어야 합니다.

아래 셀은 Docker가 있는 환경에서 현재 DB 상태를 확인합니다. Docker가 없거나 권한이 없으면 실패할 수 있습니다. 그 경우에는 터미널에서 같은 명령을 직접 실행하면 됩니다.

In [ ]:
import subprocess


def run_command(command: list[str]) -> None:
    """외부 명령을 실행하고 결과를 보기 좋게 출력합니다."""
    print("$", " ".join(command))
    completed = subprocess.run(command, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    print("exit code:", completed.returncode)


run_command(["docker", "compose", "-f", str(project_root / "database" / "docker-compose.yml"), "ps"])

In [ ]:
# PostgreSQL에 RAG 테이블이 있는지 확인합니다.
run_command([
    "docker", "exec", "maplestory-postgres",
    "psql", "-U", "admin", "-d", "mapledb",
    "-c", "select count(*) as public_table_count from information_schema.tables where table_schema = 'public';",
])

In [ ]:
# Neo4j에 그래프 노드가 있는지 확인합니다.
# 프로젝트 환경에 따라 계정이 admin 또는 neo4j일 수 있습니다.
run_command([
    "docker", "exec", "maplestory-neo4j",
    "cypher-shell", "-a", "bolt://localhost:7687",
    "-u", "neo4j", "-p", "admin123",
    "MATCH (n) RETURN count(n) AS node_count;",
])

## 8. 실제 Research Agent 실행

아래 셀은 진짜 `run_research()`를 실행합니다.

중요한 점:

- PostgreSQL 테이블과 임베딩 데이터가 없으면 DB RAG는 실패합니다.
- Web RAG는 `TAVILY_API_KEY`가 없거나 네트워크가 막혀 있으면 실패할 수 있습니다.
- Research Agent는 일부 검색이 실패해도 `errors`에 기록하고 가능한 결과를 계속 모으도록 설계되어 있습니다.

처음 공부할 때는 `auto_create_embedding=False`로 두는 것을 추천합니다. 이렇게 하면 임베딩 모델 다운로드 없이 text 검색 위주로 확인할 수 있습니다.

In [ ]:
state = {
    "user_query": "노말 스우 요구 스펙 알려줘",
    "character_name": "테스트캐릭터",
    "world_name": "스카니아",
}

# route를 직접 넣으면 어떤 검색을 쓸지 강제로 정할 수 있습니다.
# 여기서는 Web RAG를 끄고 DB + Graph만 시도합니다.
route = classify_research_route(state["user_query"])
route["use_web"] = False

result_state = run_research(
    state,
    route=route,
    auto_create_embedding=False,
)

print("retrieved_docs:", len(result_state.get("retrieved_docs", [])))
print("context length:", len(result_state.get("context", "")))
print("errors:", result_state.get("errors"))
print()
print(result_state.get("context", "")[:1500])

## 9. 무엇을 통과로 볼 것인가?

Research Agent 검증 기준은 아래처럼 잡으면 됩니다.

### 필수 통과 기준

- `classify_research_route()`가 질문 유형에 맞게 `use_graph`, `use_web`을 선택한다.
- `merge_retrieved_documents()`가 중복 문서를 제거한다.
- `build_research_context()`가 출처, 신뢰도, 검색 방식, 본문을 포함한 문자열을 만든다.
- `run_research()`가 항상 `retrieved_docs`와 `context` 키를 반환한다.

### DB 데이터가 준비된 뒤의 통과 기준

- PostgreSQL `documents`, `document_chunks`, `document_embeddings`가 존재한다.
- Neo4j node count가 0보다 크다.
- 보스/장비 질문에서 Graph 결과가 포함된다.
- 최신 이벤트/공지 질문에서 Web 결과가 포함된다.
- Final Answer Agent가 `context`만 보고 출처 있는 답변을 만들 수 있다.

## 10. 다음 연결 단계

Research Agent가 검증되면 다음 순서로 멀티에이전트 그래프에 연결하면 됩니다.

```text
Supervisor Agent
  -> 질문 의도 분석
  -> research_agent 호출
  -> retrieved_docs/context 확인
  -> final_answer 호출
  -> 사용자에게 최종 답변 반환
```

이때 Supervisor는 Research Agent 내부 구현을 몰라도 됩니다. `research_agent(state)`를 호출하면 `state`에 `retrieved_docs`와 `context`가 채워진다는 계약만 알면 됩니다.